In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
import multiprocessing
from datetime import datetime
import glob
from collections import defaultdict
from matplotlib import rc
import matplotlib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_squared_log_error

# Load graph

In [ ]:
G = nx.read_gpickle('../../legal-networks-data/de_decisions/2_network.gpickle.gz')

In [ ]:
G_md = quotient_decision_graph(G, merge_decisions=True, merge_statutes=False)
print('G_md done')
G_ms = quotient_decision_graph(G, merge_decisions=False, merge_statutes=True)
print('G_ms done')
G_md_ms = quotient_decision_graph(G, merge_decisions=True, merge_statutes=True)

In [ ]:
fG = filter_edges(G, 'edge_type', 'containment')
fG_md = filter_edges(G_md, 'Hallo 5edge_type', 'containment')
fG_ms = filter_edges(G_ms, 'edge_type', 'containment')
fG_md_ms = filter_edges(G_md_ms, 'edge_type', 'containment')

# Kookkurenzen

In [ ]:
def get_decision_nodes(G, with_root=False):
    return [
        n 
        for n, b in G.nodes(data='bipartite') 
        if b == 'decision' and (n != 'root' or with_root)
    ]


def get_decision_attr(H, node, attr):
    if '_' in node:
        decision_n = node.split('_')[0] + '_000000'
    else:
        decision_n = node
    return H.nodes[decision_n][attr]

def bin_df(source_df):
    df = source_df.groupby(['gericht', 'degree'])['node'].count().reset_index()
    df_all = source_df.groupby(['degree'])['node'].count().reset_index()
    df_all['gericht'] = 'Alle'
    df = df.append(df_all, sort=True)
    return df

def compile_df(H, filter_date='2020-01-01', additional_values=None):
    # Get decision nodes
    decision_n = get_decision_nodes(H)

    # Setup DataFrame
    degrees_df = pd.DataFrame(list(H.out_degree(decision_n)), columns=['node', 'degree'])
    degrees_df['gericht'] = [get_decision_attr(H, node=n, attr='gericht') for n in degrees_df.node]
    degrees_df['token'] = [get_decision_attr(H, node=n, attr='tokens_n') for n in degrees_df.node]
    degrees_df['datum'] = [get_decision_attr(H, node=n, attr='datum') for n in degrees_df.node]
    
    if filter_date:
        degrees_df = degrees_df[degrees_df.datum < filter_date]
        
    df = bin_df(degrees_df)
    
    if additional_values:
        for k, v in additional_values.items():
            df[k] = v
            
    return df
        
        
def plot():
    pass

In [ ]:
df = compile_df(fG, additional_values={'node_unit': 'Entscheidungsabsätze', 'degree_unit': 'Zitate (Art./§)'})
df_md = compile_df(fG_md, additional_values={'node_unit': 'Entscheidungen', 'degree_unit': 'Zitate (Art./§)'})
df_ms = compile_df(fG_ms, additional_values={'node_unit': 'Entscheidungsabsätze', 'degree_unit': 'Zitate (Gesetz)'})
df_md_ms = compile_df(fG_md_ms, additional_values={'node_unit': 'Entscheidungen', 'degree_unit': 'Zitate (Gesetz)'})
df_merged = pd.concat([df, df_md, df_ms, df_md_ms])

In [ ]:
def single_chart(df_merged, degree_unit, node_unit, show_x_axis, show_y_axis, graycolor=False):
    df = df_merged[
        (df_merged.degree_unit == degree_unit) &
        (df_merged.node_unit == node_unit) &
        (df_merged.gericht != "GmSOGB")
    ]
    
    domain_x = int(df_merged[df_merged.degree_unit == degree_unit].degree.max()) 
    domain_y = int(df_merged[df_merged.node_unit == node_unit].node.max())
    
    chart = alt.Chart(df).encode(
        alt.X(
            f'degree:Q', 
            title=degree_unit if show_x_axis else None,
            axis=alt.Axis(labels=show_x_axis, ticks=show_x_axis),
            scale=alt.Scale(domain=[0, domain_x]),
        ),
        alt.Y(
            f'node:Q',
            title=node_unit if show_y_axis else None,
            scale=alt.Scale(type='log', domain=[0.8, domain_y], nice=False),
            axis=alt.Axis(labels=show_y_axis, ticks=show_y_axis, format=".0e"),
        ),
        alt.Color(
            'gericht:N', 
            title="Gericht",
            scale=alt.Scale(range=alle_gericht_scale_range),
            legend=alt.Legend(orient='bottom', direction='horizontal', offset=5)
        ),
    )
    if graycolor:
        chart = chart.encode(
            alt.StrokeDash("gericht:N", scale=alt.Scale(range=[[1, 0], [2, 2], [6, 2]])),
            alt.Color(
                'gericht:N', 
                title="Gericht",
                scale=alt.Scale(range=['#777']*3 + ['#000']*3 + ['#AAA']*3),
                legend=alt.Legend(orient='bottom', direction='horizontal', offset=5)
            ),
        ).mark_line(size=1.8, strokeJoin='round')
    else:
        chart= chart.mark_line(strokeJoin='round')
    chart = chart.properties(height=200, width=200)
    return chart

In [ ]:
chart = alt.vconcat(
    single_chart(
        df_merged, 
        degree_unit='Zitate (Art./§)', 
        node_unit='Entscheidungsabsätze', 
        show_x_axis=False, 
        show_y_axis=True
        
    ) | single_chart(
        df_merged, 
        degree_unit='Zitate (Gesetz)', 
        node_unit='Entscheidungsabsätze', 
        show_x_axis=False, 
        show_y_axis=False
    )
,
    single_chart(
        df_merged, 
        degree_unit='Zitate (Art./§)', 
        node_unit='Entscheidungen', 
        show_x_axis=True, 
        show_y_axis=True
    ) | single_chart(
        df_merged, 
        degree_unit='Zitate (Gesetz)', 
        node_unit='Entscheidungen', 
        show_x_axis=True, 
        show_y_axis=False
        )
)
save_chart(chart, 'makro_decisions_cooccurences_combined')

In [ ]:
chart = alt.vconcat(
    single_chart(
        df_merged, 
        degree_unit='Zitate (Art./§)', 
        node_unit='Entscheidungsabsätze', 
        show_x_axis=False, 
        show_y_axis=True,
        graycolor=True,
    ) | single_chart(
        df_merged, 
        degree_unit='Zitate (Gesetz)', 
        node_unit='Entscheidungsabsätze', 
        show_x_axis=False, 
        show_y_axis=False,
        graycolor=True,
    )
,
    single_chart(
        df_merged, 
        degree_unit='Zitate (Art./§)', 
        node_unit='Entscheidungen', 
        show_x_axis=True, 
        show_y_axis=True,
        graycolor=True,
    ) | single_chart(
        df_merged, 
        degree_unit='Zitate (Gesetz)', 
        node_unit='Entscheidungen', 
        show_x_axis=True, 
        show_y_axis=False,
        graycolor=True,
        )
).configure_legend(
    strokeWidth=5,  # Adjust the stroke width of legend items
    symbolSize=190  # Adjust the size of the symbols (line length in legend)
)
save_chart(chart, 'makro_decisions_cooccurences_combined_graycolor')

In [ ]:
len(get_decision_nodes(fG))

## Weitere Statistiken

In [ ]:
df_temp = df_merged[(df_merged.node_unit == 'Entscheidungsabsätze') & (df_merged.degree_unit == 'Zitate (Art./§)') & (df_merged.gericht == 'Alle')]
assert df_temp.iloc[0].degree == 0
abs_no_ref = df_temp.iloc[0].node / df_temp.node.sum()
diss_data('meso_decision_anteil_abs_keine_ref', de_num_format(f'{abs_no_ref*100:,.2f}'))

In [ ]:
diss_data(
    'meso_decision_anteil_max_ref_art_para_in_entscheidung', 
    df_merged[(df_merged.node_unit == 'Entscheidungen') & (df_merged.degree_unit == 'Zitate (Art./§)') & (df_merged.gericht == 'Alle')].sort_values('node', ascending=False).iloc[0].degree
)
diss_data(
    'meso_decision_anteil_max_ref_gesetz_in_entscheidung', 
    df_merged[(df_merged.node_unit == 'Entscheidungen') & (df_merged.degree_unit == 'Zitate (Gesetz)') & (df_merged.gericht == 'Alle')].sort_values('node', ascending=False).iloc[0].degree
)